# GRU for Covariance Matrix


In [ ]:
import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yfinance as yf
from torch.utils.data import DataLoader

from GRUCovariance import GRUCovariance
from backtesting import run_backtest_suite
from gen_seq_data import SequenceDataset
from plots import plot_loss, plot_var
from risk_metrics import fhs_var_es
from training_functions import (
    StudentTLoss,
    gaussian_nll,
    predict_sigma,
    student_nll,
    train_covariance_model,
    train_val_test_split,
)
from vech import vech

plt.style.use("ggplot")

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

device


## Price Data


In [ ]:
macro_tickers = [
    "^GSPC",  # Equity (S&P 500)
    "GC=F",   # Gold
    "CL=F",   # Oil
    "^TNX",   # 10Y yield
]

macro_tickers


In [ ]:
macro_data = yf.download(macro_tickers, period="max")
close = macro_data["Close"].copy().dropna()

logret = np.log(close).diff()
logret["^TNX"] = close["^TNX"].diff()
logret = logret.dropna()

logret.head(), logret.tail()


In [ ]:
logret.plot(subplots=True, figsize=(12, 6), title="Log Returns / Yield Changes")
plt.tight_layout()


## Data Export & Feature Engineering


In [ ]:
project_dir = Path("/Users/bayesed/Desktop/Studium/Masterthesis/Code/neural_bekk")
data_dir = project_dir / "data"
data_dir.mkdir(exist_ok=True)

run_id = datetime.datetime.now().strftime("%Y-%m-%d")
feature_cols = macro_tickers

R = logret[feature_cols].values.astype(np.float32)
U = R[:, :, None] * R[:, None, :]
cross_returns = vech(torch.tensor(U)).detach().cpu().numpy()

input_data = np.concatenate([logret[feature_cols].values, cross_returns], axis=1)
input_df = pd.DataFrame(
    input_data,
    columns=feature_cols + [f"cross_{i+1}" for i in range(cross_returns.shape[1])],
    index=logret.index,
)

train_df, val_df, test_df = train_val_test_split(input_df)

exports = {
    "logret": logret[feature_cols],
    "train_df": train_df[feature_cols],
    "val_df": val_df[feature_cols],
    "test_df": test_df[feature_cols],
}

for name, df_export in exports.items():
    df_export = df_export.copy()
    df_export.index.name = "Date"
    df_export.to_csv(data_dir / f"{name}_{run_id}.csv")

input_df.head()


In [ ]:
pd.DataFrame(
    cross_returns,
    columns=[f"cross_{i+1}" for i in range(cross_returns.shape[1])],
    index=logret.index,
).head()


## Scaling & Sequence Dataset


In [ ]:
lookback = 60
batchsize = 128
hidden_size = 64
num_layers = 2
dropout = 0.2
epochs = 500
lr = 1e-4

mu = train_df.mean()
sigma = train_df.std()

def normalize(df, mu, sigma, with_std=True):
    if with_std:
        return (df - mu) / sigma
    return df - mu

def scale_back(sigma_scaled, sigma=None, feature_cols=None):
    if sigma is None or feature_cols is None:
        return sigma_scaled
    std_vec = sigma[feature_cols].values.astype(np.float32)
    return sigma_scaled * std_vec[None, :, None] * std_vec[None, None, :]


In [ ]:
train_norm = normalize(train_df, mu, sigma, with_std=True)
val_norm = normalize(val_df, mu, sigma, with_std=True)
test_norm = normalize(test_df, mu, sigma, with_std=True)

vol_train_ds = SequenceDataset(
    train_norm.values,
    train_norm[feature_cols].values,
    lookback,
)
vol_val_ds = SequenceDataset(
    val_norm.values,
    val_norm[feature_cols].values,
    lookback,
)
vol_test_ds = SequenceDataset(
    test_norm.values,
    test_norm[feature_cols].values,
    lookback,
)

vol_train_loader = DataLoader(vol_train_ds, batch_size=batchsize, shuffle=True)
vol_train_eval_loader = DataLoader(vol_train_ds, batch_size=batchsize, shuffle=False)
vol_val_loader = DataLoader(vol_val_ds, batch_size=batchsize, shuffle=False)
vol_test_loader = DataLoader(vol_test_ds, batch_size=batchsize, shuffle=False)

len(vol_train_ds), len(vol_val_ds), len(vol_test_ds)


## Helper Functions


In [ ]:
split_frames = {
    "train": train_df,
    "val": val_df,
    "test": test_df,
}

def fit_gru_model(loss_fn=gaussian_nll, loss_kwargs=None, epochs=epochs, lr=lr):
    k = len(feature_cols) * (len(feature_cols) + 1) // 2
    model = GRUCovariance(
        input_size=len(feature_cols) + k,
        n_assets=len(feature_cols),
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
    )

    model, history = train_covariance_model(
        model,
        vol_train_loader,
        vol_val_loader,
        loss_fn=loss_fn,
        loss_kwargs=loss_kwargs,
        epochs=epochs,
        lr=lr,
        plateau_patience=20,
        device=device,
        scheduler_type="cosine",
    )

    sigma_train_scaled, y_train_scaled = predict_sigma(model, vol_train_eval_loader, device=device)
    sigma_val_scaled, y_val_scaled = predict_sigma(model, vol_val_loader, device=device)
    sigma_test_scaled, y_test_scaled = predict_sigma(model, vol_test_loader, device=device)

    sigma_splits = {
        "train": scale_back(sigma_train_scaled, sigma=sigma, feature_cols=feature_cols),
        "val": scale_back(sigma_val_scaled, sigma=sigma, feature_cols=feature_cols),
        "test": scale_back(sigma_test_scaled, sigma=sigma, feature_cols=feature_cols),
    }
    y_splits = {
        "train": y_train_scaled,
        "val": y_val_scaled,
        "test": y_test_scaled,
    }
    return model, history, sigma_splits, y_splits

def calc_portfolio_variance(df_split, sigma_split, cols, lookback, w):
    idx = df_split.index[lookback:]
    r = df_split.loc[idx, cols].values
    r_p = (w * r).sum(axis=1)
    tmp = sigma_split @ w
    var_p = (w * tmp).sum(axis=1)
    var_p = np.clip(var_p, 1e-12, None)
    vol_p = np.sqrt(var_p)
    return idx, r_p, var_p, vol_p

def make_portfolio_df(idx, rp, varp, volp):
    return pd.DataFrame(
        {
            "r_p": np.asarray(rp, dtype=float),
            "var_p": np.asarray(varp, dtype=float),
            "vol_p": np.asarray(volp, dtype=float),
        },
        index=idx,
    )

def build_portfolio_splits(sigma_splits, w):
    portfolio_splits = {}
    for split_name, df_split in split_frames.items():
        idx, rp, varp, volp = calc_portfolio_variance(
            df_split,
            sigma_splits[split_name],
            feature_cols,
            lookback,
            w,
        )
        portfolio_splits[split_name] = make_portfolio_df(idx, rp, varp, volp)
    return portfolio_splits

def standardized_portfolio_returns(portfolio_df):
    return portfolio_df["r_p"].values / np.clip(portfolio_df["vol_p"].values, 1e-12, None)

def run_fhs_pipeline(portfolio_train, portfolio_val, portfolio_test, alpha=0.01, window=1000):
    z_train = standardized_portfolio_returns(portfolio_train)
    z_val = standardized_portfolio_returns(portfolio_val)
    z_test = standardized_portfolio_returns(portfolio_test)
    z_train_val = np.concatenate([z_train, z_val])

    train_burn_in = min(window, len(z_train) - 1)
    if train_burn_in < 1:
        raise ValueError("Training split is too short for FHS burn-in.")

    var_fhs_test, es_fhs_test, hits_test = fhs_var_es(
        z_hist_init=z_train_val,
        r_oos=portfolio_test["r_p"].values,
        vol_oos=portfolio_test["vol_p"].values,
        alpha=alpha,
        window=window,
    )
    var_fhs_val, es_fhs_val, hit_val = fhs_var_es(
        z_hist_init=z_train,
        r_oos=portfolio_val["r_p"].values,
        vol_oos=portfolio_val["vol_p"].values,
        alpha=alpha,
        window=window,
    )
    var_fhs_train, es_fhs_train, hit_train = fhs_var_es(
        z_hist_init=z_train[:train_burn_in],
        r_oos=portfolio_train["r_p"].iloc[train_burn_in:].values,
        vol_oos=portfolio_train["vol_p"].iloc[train_burn_in:].values,
        alpha=alpha,
        window=window,
    )

    return {
        "z": {
            "train": z_train,
            "val": z_val,
            "test": z_test,
            "train_val": z_train_val,
        },
        "burn_in": train_burn_in,
        "train": {"var": var_fhs_train, "es": es_fhs_train, "hits": hit_train},
        "val": {"var": var_fhs_val, "es": es_fhs_val, "hits": hit_val},
        "test": {"var": var_fhs_test, "es": es_fhs_test, "hits": hits_test},
    }

def plot_fhs_var(portfolio_df, var_fhs, alpha, title, start=0):
    idx = portfolio_df.index[start:]
    returns = portfolio_df["r_p"].iloc[start:].values
    plt.figure(figsize=(12, 4))
    plt.plot(idx, returns, color="black", lw=0.8, label="Portfolio Return")
    plt.plot(idx, var_fhs, "--", color="tab:red", lw=1.2, label=f"FHS {alpha:.1%} VaR (lower)")
    plt.fill_between(idx, var_fhs, np.zeros_like(var_fhs), color="tab:red", alpha=0.08)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


## GRU with Gaussian NLL


In [ ]:
cov_model, hist, sigma_splits, y_splits = fit_gru_model(loss_fn=gaussian_nll)

sigma_train_real = sigma_splits["train"]
sigma_val_real = sigma_splits["val"]
sigma_test_real = sigma_splits["test"]

plot_loss(hist)


In [ ]:
plot_var(alpha=0.01, lookback=lookback, cols=feature_cols, view="Train-Split (GRU Gaussian)", df=train_df, pred_vol=sigma_train_real)
plot_var(alpha=0.01, lookback=lookback, cols=feature_cols, view="Validation-Split (GRU Gaussian)", df=val_df, pred_vol=sigma_val_real)
plot_var(alpha=0.01, lookback=lookback, cols=feature_cols, view="Test-Split (GRU Gaussian)", df=test_df, pred_vol=sigma_test_real)


## Portfolio (Gaussian NLL)


In [ ]:
w = torch.full((len(feature_cols),), 1.0 / len(feature_cols), dtype=torch.float32, device=device)
w_np = w.detach().cpu().numpy()

portfolio_logrets = torch.tensor(logret[feature_cols].values, dtype=torch.float32, device=device) @ w
portfolio_logrets_np = portfolio_logrets.detach().cpu().numpy()

plt.figure(figsize=(12, 4))
plt.plot(logret.index, portfolio_logrets_np, label="Portfolio Log-Returns")
plt.title("Equal-Weight Portfolio Log-Returns")
plt.legend()
plt.tight_layout()
plt.show()

portfolio_splits = build_portfolio_splits(sigma_splits, w_np)
portfolio_train = portfolio_splits["train"]
portfolio_val = portfolio_splits["val"]
portfolio_test = portfolio_splits["test"]

portfolio_test.head()


In [ ]:
plot_var(alpha=0.01, lookback=lookback, view="Train-Split (GRU Gaussian)", portfolio=True, portfolio_df=portfolio_train)
plot_var(alpha=0.01, lookback=lookback, view="Validation-Split (GRU Gaussian)", portfolio=True, portfolio_df=portfolio_val)
plot_var(alpha=0.01, lookback=lookback, view="Test-Split (GRU Gaussian)", portfolio=True, portfolio_df=portfolio_test)


## GRU with Student-t NLL


In [ ]:
student_loss = StudentTLoss(init_nu=8.0, min_nu=2.01, max_nu=100.0)
cov_model_t, hist_t, sigma_splits_t, y_splits_t = fit_gru_model(loss_fn=student_loss)

learned_nu = float(student_loss.nu.detach().cpu())
student_kwargs = {"nu": learned_nu}

sigma_train_real_t = sigma_splits_t["train"]
sigma_val_real_t = sigma_splits_t["val"]
sigma_test_real_t = sigma_splits_t["test"]

learned_nu


In [ ]:
plot_loss(hist_t)


In [ ]:
plot_var(alpha=0.01, lookback=lookback, cols=feature_cols, view="Train-Split (GRU Student-t)", df=train_df, pred_vol=sigma_train_real_t, loss_fn=student_nll, loss_kwargs=student_kwargs)
plot_var(alpha=0.01, lookback=lookback, cols=feature_cols, view="Validation-Split (GRU Student-t)", df=val_df, pred_vol=sigma_val_real_t, loss_fn=student_nll, loss_kwargs=student_kwargs)
plot_var(alpha=0.01, lookback=lookback, cols=feature_cols, view="Test-Split (GRU Student-t)", df=test_df, pred_vol=sigma_test_real_t, loss_fn=student_nll, loss_kwargs=student_kwargs)


## Portfolio (Student-t NLL)


In [ ]:
portfolio_splits_t = build_portfolio_splits(sigma_splits_t, w_np)
portfolio_train_t = portfolio_splits_t["train"]
portfolio_val_t = portfolio_splits_t["val"]
portfolio_test_t = portfolio_splits_t["test"]

portfolio_test_t.head()


In [ ]:
plot_var(alpha=0.01, lookback=lookback, view="Train-Split (GRU Student-t)", portfolio=True, portfolio_df=portfolio_train_t, loss_fn=student_nll, loss_kwargs=student_kwargs)
plot_var(alpha=0.01, lookback=lookback, view="Validation-Split (GRU Student-t)", portfolio=True, portfolio_df=portfolio_val_t, loss_fn=student_nll, loss_kwargs=student_kwargs)
plot_var(alpha=0.01, lookback=lookback, view="Test-Split (GRU Student-t)", portfolio=True, portfolio_df=portfolio_test_t, loss_fn=student_nll, loss_kwargs=student_kwargs)


## Filtered Historical Simulation


In [ ]:
alpha = 0.01
window = 1000

fhs_normal = run_fhs_pipeline(portfolio_train, portfolio_val, portfolio_test, alpha=alpha, window=window)
var_fhs_train = fhs_normal["train"]["var"]
es_fhs_train = fhs_normal["train"]["es"]
hit_train = fhs_normal["train"]["hits"]

var_fhs_val = fhs_normal["val"]["var"]
es_fhs_val = fhs_normal["val"]["es"]
hit_val = fhs_normal["val"]["hits"]

var_fhs_test = fhs_normal["test"]["var"]
es_fhs_test = fhs_normal["test"]["es"]
hits_test = fhs_normal["test"]["hits"]

pd.DataFrame(
    {
        "hit_rate": [hit_train.mean(), hit_val.mean(), hits_test.mean()],
    },
    index=["train", "val", "test"],
)


In [ ]:
plot_fhs_var(
    portfolio_train,
    var_fhs_train,
    alpha=alpha,
    title="Filtered Historical Simulation VaR (GRU Gaussian, Train)",
    start=fhs_normal["burn_in"],
)
plot_fhs_var(
    portfolio_val,
    var_fhs_val,
    alpha=alpha,
    title="Filtered Historical Simulation VaR (GRU Gaussian, Validation)",
)
plot_fhs_var(
    portfolio_test,
    var_fhs_test,
    alpha=alpha,
    title="Filtered Historical Simulation VaR (GRU Gaussian, Test)",
)


In [ ]:
fhs_student = run_fhs_pipeline(portfolio_train_t, portfolio_val_t, portfolio_test_t, alpha=alpha, window=window)
var_fhs_train_t = fhs_student["train"]["var"]
es_fhs_train_t = fhs_student["train"]["es"]
hit_train_t = fhs_student["train"]["hits"]

var_fhs_val_t = fhs_student["val"]["var"]
es_fhs_val_t = fhs_student["val"]["es"]
hit_val_t = fhs_student["val"]["hits"]

var_fhs_test_t = fhs_student["test"]["var"]
es_fhs_test_t = fhs_student["test"]["es"]
hits_test_t = fhs_student["test"]["hits"]

pd.DataFrame(
    {
        "hit_rate": [hit_train_t.mean(), hit_val_t.mean(), hits_test_t.mean()],
    },
    index=["train", "val", "test"],
)


In [ ]:
plot_fhs_var(
    portfolio_train_t,
    var_fhs_train_t,
    alpha=alpha,
    title="Filtered Historical Simulation VaR (GRU Student-t, Train)",
    start=fhs_student["burn_in"],
)
plot_fhs_var(
    portfolio_val_t,
    var_fhs_val_t,
    alpha=alpha,
    title="Filtered Historical Simulation VaR (GRU Student-t, Validation)",
)
plot_fhs_var(
    portfolio_test_t,
    var_fhs_test_t,
    alpha=alpha,
    title="Filtered Historical Simulation VaR (GRU Student-t, Test)",
)


## Backtesting


In [ ]:
results = None
results_student = None

try:
    results = run_backtest_suite(
        returns=portfolio_test["r_p"].values,
        var=var_fhs_test,
        es=es_fhs_test,
        alpha=alpha,
        volatility=portfolio_test["vol_p"].values,
    )

    results_student = run_backtest_suite(
        returns=portfolio_test_t["r_p"].values,
        var=var_fhs_test_t,
        es=es_fhs_test_t,
        alpha=alpha,
        volatility=portfolio_test_t["vol_p"].values,
    )
except ImportError as exc:
    print(f"Backtesting dependencies are missing: {exc}")

{
    "gru_gaussian": results,
    "gru_student_t": results_student,
}
